# System Metrics Study

In [239]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

## Experiment Configuration

In [240]:
dataset = "aitv2"
scenario = "santos"

In [241]:
# dataset = "darpa2000"
# scenario = "s1_inside"

In [242]:
experiment_dir = f"../../experiments/{dataset}/{scenario}/logic_study/deepproblog"

In [243]:
experiment = "system_metrics_study"
plots_dir = Path(f"../../reports/{dataset}/{scenario}/{experiment}")
plots_dir.mkdir(parents=True, exist_ok=True)

## Inference Times

In [244]:
metrics_dir = Path(f"{experiment_dir}/metrics")
file_paths = list(metrics_dir.glob("*.npz"))
    
grouped_times = defaultdict(list)
for file_path in file_paths:
    full_name = file_path.stem
    base_name = full_name.rsplit('_', 2)[0]
    data = np.load(file_path, allow_pickle=True)
    grouped_times[base_name].extend(data["inference_times"].tolist())


inference_stats = {}
for base_name, all_times in grouped_times.items():
    avg_time = np.mean(all_times)
    std_time = np.std(all_times)
    
    print(f"{base_name}:")
    print(f"  Runs pooled: {len(all_times)} total inference steps")
    print(f"  Average Time: {avg_time:.6f} seconds")
    print(f"  Std Dev Time: {std_time:.6f} seconds\n")
    
    inference_stats[base_name] = {
        "avg_inference_time": avg_time,
        "std_inference_time": std_time
    }


ait_temp_context_pretrained_base_w100_full_0.001lr:
  Runs pooled: 528980 total inference steps
  Average Time: 0.000464 seconds
  Std Dev Time: 0.003316 seconds

ait_temp_pretrained_base_w100_full_0.001lr:
  Runs pooled: 211592 total inference steps
  Average Time: 0.002584 seconds
  Std Dev Time: 0.049321 seconds

ait_temp_endtoend_base_w100_full_0.001lr:
  Runs pooled: 528980 total inference steps
  Average Time: 0.002413 seconds
  Std Dev Time: 0.036930 seconds

ait_temp_context_endtoend_base_w100_full_0.001lr:
  Runs pooled: 528980 total inference steps
  Average Time: 0.000465 seconds
  Std Dev Time: 0.001095 seconds

ait_temp_baseline_endtoend_base_w100_full_0.001lr:
  Runs pooled: 105796 total inference steps
  Average Time: 0.000105 seconds
  Std Dev Time: 0.000245 seconds

ait_temp_context_baseline_endtoend_base_w100_full_0.001lr:
  Runs pooled: 105796 total inference steps
  Average Time: 0.000155 seconds
  Std Dev Time: 0.006782 seconds



In [245]:
inference_stats_df = pd.DataFrame.from_dict(inference_stats, orient='index')
inference_stats_df.to_csv(plots_dir / "inference_time_stats.csv")

In [246]:
inference_stats_df

,avg_inference_time,std_inference_time
ait_temp_context_pretrained_base_w100_full_0.001lr,0.000464,0.003316
ait_temp_pretrained_base_w100_full_0.001lr,0.002584,0.049321
ait_temp_endtoend_base_w100_full_0.001lr,0.002413,0.036930
ait_temp_context_endtoend_base_w100_full_0.001lr,0.000465,0.001095
ait_temp_baseline_endtoend_base_w100_full_0.001lr,0.000105,0.000245
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr,0.000155,0.006782


## Memory Usage

In [247]:
def load_monitor_csv(file_path):
    try:
        df = pd.read_csv(file_path, parse_dates=['ts'], date_format='%Y-%m-%d %H:%M:%S')
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        df = pd.DataFrame()
    
    if 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], format='%Y-%m-%dT%H:%M:%S.%f+00:00Z')
        df['t_s'] = (df['ts'] - df['ts'].iloc[0]).dt.total_seconds()
    else:
        print(f"'ts' column not found in {file_path}.")
    
    return df

In [248]:
def summarize_memory(df, col='rss_mb'):
    secs = df['t_s'].values
    vals = df[col].values
    peak = np.nanmax(vals)
    peak_idx = np.nanargmax(vals)
    time_to_peak = secs[peak_idx]
    auc = np.trapezoid(vals, secs)  # MB·s
    median = np.nanmedian(vals)
    p90 = np.nanpercentile(vals, 90)
    p99 = np.nanpercentile(vals, 99)
    # linear slope MB/s (robust: use first N seconds to estimate steady growth)
    if len(secs) > 5:
        coeff = np.polyfit(secs, vals, 1)[0]
    else:
        coeff = np.nan
    return {
        'peak_mb': float(peak),
        'time_to_peak_s': float(time_to_peak),
        'auc_mb_s': float(auc),
        'median_mb': float(median),
        'p90_mb': float(p90),
        'p99_mb': float(p99),
        'slope_mb_per_s': float(coeff),
    }

In [249]:
# Group records by (model_name, phase) to separate train and test runs
monitor_dir = Path(f"{experiment_dir}/monitor")
file_paths = list(monitor_dir.glob("*.csv"))

grouped_records = defaultdict(list)

for file_path in file_paths:
    experiment_name = file_path.stem
    parts = experiment_name.split("_")
    phase = parts[-2]  # "train" or "test"
    model_name = "_".join(parts[1:-8])
    print(f"Loading {model_name} ({phase})...")
    
    df = load_monitor_csv(file_path)
    summary = summarize_memory(df)
    grouped_records[(model_name, phase)].append(summary)

# Compute averages and std across runs
records = []
for (model_name, phase), summaries in grouped_records.items():
    
    aggregated_metrics = {
        "Model": model_name,
        "Phase": phase,
        "runs": len(summaries)  # Keep track of how many runs were averaged
    }
    
    # Calculate mean and std for every metric returned by summarize_memory()
    for key in summaries[0].keys():
        values = [s[key] for s in summaries]
        aggregated_metrics[f"{key}_mean"] = np.mean(values)
        aggregated_metrics[f"{key}_std"] = np.std(values)
        
    records.append(aggregated_metrics)

# Build DataFrame (no longer using from_dict with orient="index" since we made a list)
results_df = pd.DataFrame(records)

# Optional: format for readability based on the new "_mean" keys
def format_duration(seconds):
    if seconds >= 3600:
        return f"{seconds/3600:.1f}h"
    elif seconds >= 60:
        return f"{seconds/60:.1f}m"
    return f"{seconds:.0f}s"

# Note the updated keys targeting the "_mean" columns
results_df["time_to_peak"] = results_df["time_to_peak_s_mean"].apply(format_duration)
results_df["peak_gb"] = (results_df["peak_mb_mean"] / 1024).round(2)
results_df["peak_gb_std"] = (results_df["peak_mb_std"] / 1024).round(2)
results_df["median_gb"] = (results_df["median_mb_mean"] / 1024).round(2)

# Pivot or display side-by-side
display_cols = [
    "Model", "Phase", "runs", 
    "peak_gb", "peak_gb_std", 
    "median_gb", "time_to_peak", 
    "slope_mb_per_s_mean"
]
print(results_df[display_cols])


Loading temp_context_endtoend (train)...
Loading temp_endtoend (test)...
Loading temp_endtoend (train)...
Loading temp_endtoend (train)...
Loading temp_context_pretrained (test)...
Loading temp_context_endtoend (train)...
Loading temp_context_endtoend (test)...
Loading temp_endtoend (train)...
Loading temp_context_pretrained (test)...
Loading temp_context_endtoend (train)...
Loading temp_endtoend (train)...
Loading temp_context_endtoend (test)...
Loading temp_pretrained (train)...
Loading temp_context_pretrained (train)...
Loading temp_context_pretrained (test)...
Loading temp_context_endtoend (train)...
Loading temp_context_pretrained (test)...
Loading temp_endtoend (test)...
Loading temp_endtoend (test)...
Loading temp_context_pretrained (train)...
Loading temp_context_pretrained (train)...
Loading temp_context_pretrained (train)...
Loading temp_pretrained (test)...
Loading temp_pretrained (train)...
Loading temp_context_endtoend (test)...
Loading temp_context_endtoend (test)...
Load

In [250]:
results_df.to_csv(f"{plots_dir}/system_metrics_summary.csv", index=False)

In [251]:
results_df[results_df["Phase"] == "test"]


,Model,Phase,runs,peak_mb_mean,peak_mb_std,time_to_peak_s_mean,time_to_peak_s_std,auc_mb_s_mean,auc_mb_s_std,median_mb_mean,...,p90_mb_mean,p90_mb_std,p99_mb_mean,p99_mb_std,slope_mb_per_s_mean,slope_mb_per_s_std,time_to_peak,peak_gb,peak_gb_std,median_gb
1,temp_endtoend,test,4,30543.154297,573.801722,221.747172,40.248913,6.686158e+06,1.089061e+06,30214.628906,...,30464.458008,658.151093,30540.883574,574.353807,1.623985,0.455495,3.7m,29.83,0.56,29.51
3,temp_context_pretrained,test,4,30918.687500,5.658096,55.463660,2.362292,1.698229e+06,7.248896e+04,30591.663574,...,30867.629004,21.050528,30916.781816,5.800600,8.182023,0.317422,55s,30.19,0.01,29.87
4,temp_context_endtoend,test,4,30918.519531,2.800922,58.628720,1.093574,1.795355e+06,3.376553e+04,30593.239746,...,30879.670020,15.418563,30916.433135,2.760146,8.087944,0.305897,59s,30.19,0.00,29.88
7,temp_pretrained,test,1,25937.164062,0.000000,196.682442,0.000000,5.042943e+06,0.000000e+00,25610.509766,...,25913.746094,0.000000,25934.953125,0.000000,1.829116,0.000000,3.3m,25.33,0.00,25.01


In [252]:
results_df[results_df["Phase"] == "test"]["peak_gb_std"]

1    0.56
3    0.01
4    0.00
7    0.00
Name: peak_gb_std, dtype: float64